# Custom Kion TTS Dataset Generator
This notebook uses IndexTTS2 to generate synthetic training data for a final TTS model.
It supports:
- **Single-style**: `[sarcastic=0.7] Oh, brilliant.`
- **Emotion blends**: `[playful=0.7,teasing=0.5] You actually did it?`
- **Style transitions**: `[sarcastic=0.7] Oh, fantastic. [concerned=0.6] Are you okay?`

## 1. Setup & Installation

In [ ]:
!git clone https://github.com/index-tts/index-tts.git
%cd index-tts
!pip install -e .
!pip install pydub  # Used for audio crossfading

from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download(repo_id="IndexTeam/IndexTTS-2", local_dir="/content/index-tts/checkpoints")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile
from google.colab import files
import shutil

# Ensure directories exist
os.makedirs('/content/kionRefVoice', exist_ok=True)
os.makedirs('/content/EmotionRefs', exist_ok=True)

print("1. Handling main speaker reference file:")
drive_spk_path = '/content/drive/MyDrive/kion_reference.wav'
local_spk_path = '/content/kionRefVoice/kion_reference.wav'

if os.path.exists(drive_spk_path):
    print(f"Found {drive_spk_path} in Google Drive. Copying...")
    shutil.copy(drive_spk_path, local_spk_path)
    print(f"Copied to {local_spk_path}")
else:
    print("kion_reference.wav not found in Drive. Please upload it manually:")
    uploaded_spk = files.upload()
    for filename in uploaded_spk.keys():
        os.rename(filename, local_spk_path)
        print(f"Moved {filename} to {local_spk_path}")

print("\n2. Handling emotion reference files (e.g., zip file from Drive):")
drive_zip_path = '/content/drive/MyDrive/emo_refs.zip'
if os.path.exists(drive_zip_path):
    print(f"Found {drive_zip_path} in Google Drive. Copying and extracting...")
    local_zip_path = '/content/emo_refs.zip'
    shutil.copy(drive_zip_path, local_zip_path)

    if os.path.exists(local_zip_path):
        print(f"Unzipping {local_zip_path} to /content/EmotionRefs/")
        with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/EmotionRefs')
        os.remove(local_zip_path)
        print(f"Extracted and cleaned up {local_zip_path}")
else:
    print("No emo_refs.zip found in Google Drive. Please upload emotion reference files (wavs or zip).")
    uploaded_emo = files.upload()
    for filename in uploaded_emo.keys():
        if filename.endswith('.zip'):
            print(f"Unzipping {filename} to /content/EmotionRefs/")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/EmotionRefs')
            os.remove(filename)
            print(f"Extracted and cleaned up {filename}")
        else:
            os.rename(filename, os.path.join('/content/EmotionRefs', filename))
            print(f"Moved {filename} to /content/EmotionRefs/")

In [ ]:
import os
import subprocess
import sys

print("Reinstalling stable torch suite to resolve C++ extension (NMS) linkage...")

# Force uninstall to clear broken links
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])

# Install stable versions compatible with Colab CUDA environment
# 2.4.0 is the current standard that fixes the registration errors
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", "torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0"])

print("\nInstallation complete. RESTARTING RUNTIME to finalize operator registration...")

In [ ]:
# Fix Protobuf AttributeError and Numpy/Numba version conflict
!pip install "protobuf==3.20.3" "numpy==2.2.6" --force-reinstall

## 2. Load Model

In [ ]:
import sys
import os
import torch

sys.path.append("/content/index-tts")
from indextts.infer_v2 import IndexTTS2

tts = IndexTTS2(
    cfg_path="/content/index-tts/checkpoints/config.yaml",
    model_dir="/content/index-tts/checkpoints",
    use_fp16=torch.cuda.is_available(),
    use_cuda_kernel=False,
)
print("IndexTTS2 loaded.")

## 3. Define the Style Parser

In [ ]:
import re

def parse_inline_tags(text):
    """
    Parses: `[playful=0.7,teasing=0.5] Oh, you actually managed to do it?`
    Returns: [
        {'styles': {'playful': 0.7, 'teasing': 0.5}, 'text': 'Oh, you actually managed to do it?'}
    ]
    """
    # If there are no brackets at all, treat the entire string as neutral
    if '[' not in text and ']' not in text:
        return [{"styles": {}, "text": text.strip()}]

    pattern = r'\[(.*?)\]\s*(.*?)(?=\[|$)'
    matches = re.findall(pattern, text)

    segments = []
    for tag_str, content in matches:
        styles = {}
        for style_pair in tag_str.split(','):
            if '=' in style_pair:
                k, v = style_pair.split('=')
                styles[k.strip()] = float(v.strip())
            elif ':' in style_pair:
                k, v = style_pair.split(':')
                styles[k.strip()] = float(v.strip())

        segments.append({
            "styles": styles,
            "text": content.strip()
        })

    return segments

print("Testing parser (neutral):", parse_inline_tags("Just a normal sentence."))
print("Testing parser (tagged):", parse_inline_tags("[dry-sarcasm=0.7] Oh, fantastic. [concerned-mild=0.6] Are you okay?"))


## 4. Dataset Generation Pipeline
This handles single-style, blends, and transitions (via AudioSegment crossfading).

In [ ]:
import uuid
from pydub import AudioSegment
from IPython.display import Audio, display
import shutil
import os

os.makedirs("/content/dataset/wavs", exist_ok=True)
os.makedirs("/content/dataset/temp", exist_ok=True)

KION_REF = "/content/kionRefVoice/kion_reference.wav"
EMO_REFS_DIR = "/content/EmotionRefs"


EMOTIONS_SET = {"angry", "annoyed", "bored", "concerned", "confused", "curious", "disappointed", "excited", "frustrated", "happy", "heartbroken", "overjoyed", "sad", "surprised"}
STYLES_SET = {"affectionate", "authoritative", "calm", "deadpan", "dramatic", "playful", "sarcasm", "serious", "soothing", "teasing"}

def split_metadata_tags(styles_dict):
    emotions = {}
    styles = {}
    for k, v in styles_dict.items():
        if k in EMOTIONS_SET:
            emotions[k] = v
        elif k in STYLES_SET:
            styles[k] = v
        else:
            styles[k] = v
    return emotions, styles

def get_dominant_style(styles):
    if not styles: return None, 0.0
    return max(styles.items(), key=lambda x: x[1])

def get_audio_ref(base_emotion, intensity):
    mapping = {
        "angry": [(0.6, "angry-mild"), (1.1, "angry-strong")],
        "annoyed": [(0.6, "annoyed-mild"), (1.1, "annoyed-strong")],
        "concerned": [(0.6, "concerned-mild"), (1.1, "concerned-strong")],
        "confused": [(0.6, "confused-mild"), (1.1, "confused-strong")],
        "curious": [(0.6, "curious-mild"), (1.1, "curious-strong")],
        "disappointed": [(0.5, "disappointed-mild"), (0.7, "disappointed-medium"), (1.1, "disappointed-strong")],
        "dramatic": [(0.6, "dramatic-medium"), (1.1, "dramatic-strong")],
        "excited": [(0.6, "excited-medium"), (1.1, "excited-strong")],
        "frustrated": [(0.6, "frustrated-mild"), (1.1, "frustrated-strong")],
        "happy": [(0.6, "happy-mild"), (1.1, "happy-strong")],
        "sad": [(0.6, "sad-mild"), (1.1, "sad-strong")],
        "suprised": [(0.6, "surprised-mild"), (1.1, "surprised-strong")],
        "surprised": [(0.6, "surprised-mild"), (1.1, "surprised-strong")],
        "sarcasm": [(1.1, "dry-sarcasm")]
    }
    if base_emotion in mapping:
        for threshold, name in mapping[base_emotion]:
            if intensity < threshold:
                return f"voice_preview_{name}.wav"
    return f"voice_preview_{base_emotion}.wav"

def get_compound_audio_ref(styles_dict):
    try:
        available_refs = os.listdir(EMO_REFS_DIR)
    except:
        return None

    expected_parts = []
    for k, v in styles_dict.items():
        norm_k = 'heartbroken' if k == 'heartbreak' else 'surprised' if k == 'suprised' else 'sarcasm' if k == 'sarcastic' else k
        val_str = str(v).replace('.', '-')
        expected_parts.append(f"{norm_k}{val_str}")

    import itertools
    for file in available_refs:
        if file.startswith("voice_preview_") and file.endswith(".wav"):
            basename = file.replace("voice_preview_", "").replace(".wav", "")
            for perm in itertools.permutations(expected_parts):
                if basename == "-".join(perm):
                    return file
    return None

def generate_segment(text, styles_dict, temp_path):
    if not styles_dict:
        print(f"Using neutral baseline for text: '{text[:20]}...'")
        tts.infer(
            spk_audio_prompt=KION_REF,
            text=text,
            output_path=temp_path,
            verbose=False
        )
        return

    if len(styles_dict) > 1:
        compound_ref = get_compound_audio_ref(styles_dict)
        if compound_ref:
            expected_ref = os.path.join(EMO_REFS_DIR, compound_ref)
            dominant_emotion, intensity = get_dominant_style(styles_dict)
            print(f"Using EXACT COMPOUND audio ref {expected_ref}")
            tts.infer(
                spk_audio_prompt=KION_REF,
                text=text,
                emo_audio_prompt=expected_ref,
                emo_alpha=intensity,
                output_path=temp_path,
                verbose=False
            )
            return
        else:
            dominant_emotion, intensity = get_dominant_style(styles_dict)
            blend_desc = " and ".join([f"{k}" for k in styles_dict.keys()])
            print(f"No compound ref match, using text blend '{blend_desc}'")
            tts.infer(
                spk_audio_prompt=KION_REF,
                text=text,
                use_emo_text=True,
                emo_text=f"{blend_desc} delivery",
                emo_alpha=intensity,
                output_path=temp_path,
                verbose=False
            )
            return

    dominant_emotion, intensity = get_dominant_style(styles_dict)
    ref_filename = get_audio_ref(dominant_emotion, intensity)
    expected_ref = os.path.join(EMO_REFS_DIR, ref_filename)

    if os.path.exists(expected_ref):
        print(f"Using audio ref {expected_ref} for style {dominant_emotion} (intensity {intensity})")
        tts.infer(
            spk_audio_prompt=KION_REF,
            text=text,
            emo_audio_prompt=expected_ref,
            emo_alpha=intensity,
            output_path=temp_path,
            verbose=False
        )
    else:
        print(f"Audio ref {ref_filename} not found, falling back to text-based.")
        tts.infer(
            spk_audio_prompt=KION_REF,
            text=text,
            use_emo_text=True,
            emo_text=f"{dominant_emotion} delivery",
            emo_alpha=intensity,
            output_path=temp_path,
            verbose=False
        )

def process_utterance(full_text):
    segments = parse_inline_tags(full_text)
    if not segments:
        return

    sample_id = str(uuid.uuid4())[:8]
    final_wav_path = f"/content/dataset/wavs/{sample_id}.wav"

    temp_files = []
    metadata_segments = []

    for i, seg in enumerate(segments):
        temp_path = f"/content/dataset/temp/{sample_id}_seg{i}.wav"
        generate_segment(seg['text'], seg['styles'], temp_path)
        temp_files.append(temp_path)

        emotions, styles = split_metadata_tags(seg['styles'])
        metadata_segments.append({
            "text": seg['text'],
            "emotions": emotions,
            "styles": styles
        })

    if len(temp_files) == 1:
        shutil.copy(temp_files[0], final_wav_path)
    else:
        combined = AudioSegment.from_wav(temp_files[0])
        for path in temp_files[1:]:
            next_seg = AudioSegment.from_wav(path)
            combined = combined.append(next_seg, crossfade=80)
        combined.export(final_wav_path, format="wav")

    dataset_metadata.append({
        "id": sample_id,
        "text": full_text,
        "segments": metadata_segments,
        "speaker_reference": "kion_reference.wav",
        "wav_path": final_wav_path
    })

    print(f"Generated: {final_wav_path}")
    display(Audio(final_wav_path))



## 5. Run the Generation

In [ ]:
import os
import json
import shutil
import zipfile

# --- DISTRIBUTED GENERATION CONFIG ---
BATCH_SIZE = 10
# Define the start and end batch numbers this specific session is responsible for.
# Example: (151, 200) for session 1, (201, 250) for session 2, etc.
BATCHES_RESPONSIBLE = (1501, 1550)
START_BATCH_ID, END_BATCH_ID = BATCHES_RESPONSIBLE

# Dynamically create an isolated folder name based on the assigned batches
BASE_DRIVE_DIR = "/content/drive/MyDrive/KionTTS_Dataset"
DRIVE_OUTPUT_DIR = os.path.join(BASE_DRIVE_DIR, f"Batch{START_BATCH_ID}-{END_BATCH_ID}")

drive_bank_path = "/content/drive/MyDrive/sentence_bank.txt"
local_bank_path = "/content/dataset/sentence_bank.txt"

# Create necessary directories
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs("/content/dataset", exist_ok=True)

# 1. Load the Sentence Bank
bank_path = drive_bank_path if os.path.exists(drive_bank_path) else local_bank_path
if not os.path.exists(bank_path):
    raise FileNotFoundError(f"Sentence bank not found at {bank_path}")

with open(bank_path, "r") as f:
    sentence_bank = [line.strip() for line in f.readlines() if line.strip()]
total_sentences = len(sentence_bank)

# Helper function to calculate exact index based on batch history
def get_expected_index(b_id):
    if b_id == 0:
        return 0
    return 100 + (b_id - 1) * BATCH_SIZE

# 2. Simplified Resumption Logic (Isolated Folder)
existing_zips = sorted([f for f in os.listdir(DRIVE_OUTPUT_DIR) if f.startswith("batch_") and f.endswith(".zip")])

if existing_zips:
    latest_zip_name = existing_zips[-1]
    latest_zip_path = os.path.join(DRIVE_OUTPUT_DIR, latest_zip_name)

    # Extract the exact batch ID from the file name
    latest_batch_id = int(latest_zip_name.replace("batch_", "").replace(".zip", ""))
    batch_counter = latest_batch_id + 1

    try:
        with zipfile.ZipFile(latest_zip_path, 'r') as z:
            with z.open('metadata.json') as f:
                last_metadata = json.load(f)
                if last_metadata:
                    last_text = last_metadata[-1]['text']
                    try:
                        last_found_idx = sentence_bank.index(last_text)
                        current_idx = last_found_idx + 1
                        print(f"Detected latest batch: {latest_zip_name}. Last sentence matched at index {last_found_idx}.")
                    except ValueError:
                        print(f"Warning: Last sentence in metadata not found in bank. Defaulting to math-based index.")
                        current_idx = get_expected_index(batch_counter)
    except Exception as e:
        print(f"Could not read metadata from zip: {e}. Falling back to manual calculation.")
        current_idx = get_expected_index(batch_counter)
else:
    # No batches in this specific folder yet. Start from assigned minimum.
    batch_counter = START_BATCH_ID
    current_idx = get_expected_index(batch_counter)
    print(f"No existing batches found in {DRIVE_OUTPUT_DIR}. Starting fresh.")

print(f"Session responsible for batches {START_BATCH_ID} to {END_BATCH_ID}.")
print(f"Resuming from sentence index {current_idx} into batch_{batch_counter:04d}...")

# 3. Main Loop
while current_idx < total_sentences and batch_counter <= END_BATCH_ID:
    # If by some chance this code runs batch 0, make sure it pulls 100 instead of 10
    current_batch_size = 100 if batch_counter == 0 else BATCH_SIZE

    start_idx = current_idx
    end_idx = min(start_idx + current_batch_size, total_sentences)
    batch_sentences = sentence_bank[start_idx:end_idx]
    batch_name = f"batch_{batch_counter:04d}"

    print(f"\n--- Processing {batch_name} (Sentences {start_idx} to {end_idx-1}) ---")

    shutil.rmtree("/content/dataset/wavs", ignore_errors=True)
    shutil.rmtree("/content/dataset/temp", ignore_errors=True)
    os.makedirs("/content/dataset/wavs", exist_ok=True)
    os.makedirs("/content/dataset/temp", exist_ok=True)

    dataset_metadata = []
    for text in batch_sentences:
        # NOTE: Ensure process_utterance appends to dataset_metadata
        process_utterance(text)

    with open("/content/dataset/metadata.json", "w") as f:
        json.dump(dataset_metadata, f, indent=2)

    # Save to Colab temporarily, then copy to the isolated Drive folder
    shutil.make_archive(f"/content/{batch_name}", 'zip', "/content/dataset")
    shutil.copy(f"/content/{batch_name}.zip", os.path.join(DRIVE_OUTPUT_DIR, f"{batch_name}.zip"))
    os.remove(f"/content/{batch_name}.zip")

    current_idx = end_idx
    batch_counter += 1

if batch_counter > END_BATCH_ID:
    print(f"\nAll assigned batches ({START_BATCH_ID} to {END_BATCH_ID}) completed successfully!")
else:
    print("\nFinished processing all available sentences in the bank!")